# Начало

В данном ноутбуке я распрашиваю pdf файлы стратегий университетов из папки ``/Стратегии`` и анотирую их.

Загрузим библиотеки

In [1]:
from pypdf import PdfReader # для распаршивания пдф
import re # для работы с подстроками и строками
import pandas as pd # для работы с таблицами
import nltk # для загрузки токенезатора
from pathlib import Path # Для работы с путями
from tqdm.autonotebook import tqdm # для прогресс баров
from nltk.tokenize import sent_tokenize # для токеназации текстов
from openai import OpenAI # для аннотации данных 
import time # для работы с таймерами

C:\Users\kiril\AppData\Local\Temp\ipykernel_26976\4124360600.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm # для прогресс баров


Определим функцию, которая построчно считывает pdf и перерводит его в текст.

In [2]:
def get_text_from_strategy_pdf(file_name: str):
    reader = PdfReader(file_name)
    text = "".join(reader.pages[i].extract_text() or "" for i in range(1, len(reader.pages)))

    text = re.sub(r'(?<=\w)-\s*\n\s*(?=\w)', '', text)

    text = re.sub(r'\n{2,}', '\n', text)   
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s{2,}',  ' ', text)
    text = text.lower()
    
    return text.strip()

Загружаем список всех стратегий

In [3]:
doc_paths = sorted(Path("Стратегии").glob("*.pdf"))
print(f'Загружено {len(doc_paths)} стратегий:')
doc_paths

Загружено 20 стратегий:


[WindowsPath('Стратегии/BOCCONI UNIVERSITY STRATEGY.pdf'),
 WindowsPath('Стратегии/CAIP TOWN UNIVERSITY STRATEGY.pdf'),
 WindowsPath('Стратегии/COPENHAGEN UNIVERSITY STRATEGY.pdf'),
 WindowsPath('Стратегии/Dawson College.pdf'),
 WindowsPath('Стратегии/Florida International University.pdf'),
 WindowsPath('Стратегии/King’s University College.pdf'),
 WindowsPath('Стратегии/Lakehead University.pdf'),
 WindowsPath('Стратегии/LONDON SCHOOL OF ECONOMICS STRATEGY.pdf'),
 WindowsPath('Стратегии/Montclair State University.pdf'),
 WindowsPath('Стратегии/Saint Mary’s University.pdf'),
 WindowsPath('Стратегии/STANFORD UNIVERSITY STRATEGY.pdf'),
 WindowsPath('Стратегии/Texas Tech University.pdf'),
 WindowsPath('Стратегии/University of Calgaryl.pdf'),
 WindowsPath('Стратегии/University of Connecticut.pdf'),
 WindowsPath('Стратегии/University of Georgia.pdf'),
 WindowsPath('Стратегии/University of Maine at Augusta.pdf'),
 WindowsPath('Стратегии/university of manitoba.pdf'),
 WindowsPath('Стратегии/Uni

Вычилиняем текст из всех стратегий

In [67]:
texts = [get_text_from_strategy_pdf(p) for p in tqdm(doc_paths)]
texts

  0%|          | 0/20 [00:00<?, ?it/s]

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)


['introduction 4 1. the demographic and economic scenario 14 2. the job market: trends, challenges and opportunities 24 part 2 3. the competitive landscape: trends, challenges and opportunities 32 3.1 the main trends affecting the demand for higher education 34 3.2 main trends affecting the supply of higher education 39 3.3 the relevance of the european dimension 50 part 1 4. strategic goals driving vision 2030 56 1. a leading independent international university in the social sciences 58 2. a university committed to the advancement of knowledge through rigorous and relevant research 59 3. a university offering a life transforming learning experience of the highest academic quality 60 4. a university promoting impact and engagement of all its stakeholders 62 5. an open university supporting social mobility, inclusivity, diversity and sustainability 63 6. a university promoting innovation and entrepreneurship 64 5. strategic plan 2021-2025 66 1. main quantitative targets for 2025 68 2. 

Делаем фильтрацию по строкам, методология описана в эссе и сохраняем валидные предложения в DataFrame

In [ ]:
# nltk.download('punkt')

def is_valid_sentence(sent):
    # Убираем не-буквенные символы
    sent_clean = re.sub(r"[^a-zA-Z ]+", "", sent)
    words = sent_clean.split()
    
    # Фильтрация по длине и наличию глаголов
    if len(words) <= 10:
        return False
    
    if not re.search(r'\b(is|are|has|have|will|aim|support|contribute|engage|ensure|promote|commit|advance|enable|serve|provide)\b', sent, re.IGNORECASE):
        return False
    
    # Убираем предложения, начинающиеся с маркеров
    if sent.strip().startswith(('●', '-', '■', '>', '/', '*')):
        return False

    num_count = sum(c.isdigit() for c in sent)
    word_count = len(words)
    short_word_count = sum(len(w) <= 2 for w in words)
    
    if num_count > word_count * 0.2:
        return False
    if short_word_count > word_count * 0.5:
        return False
    return True

def extract_valid_sentences(texts, doc_paths):
    all_sentences = []
    for text, path in tqdm(zip(texts, doc_paths)):
        sentences = sent_tokenize(text)
        for sent in sentences:
            if is_valid_sentence(sent):
                all_sentences.append({
                    "sentence": sent.strip(),
                    "university": path.stem
                })
    return pd.DataFrame(all_sentences)


df = extract_valid_sentences(texts, doc_paths)

0it [00:00, ?it/s]

Выведем получивщийся датафрейм

In [69]:
df

,sentence,university
0,demand factors are affected by the increasing ...,BOCCONI UNIVERSITY STRATEGY
1,the launch of entire online programs from some...,BOCCONI UNIVERSITY STRATEGY
2,the fact that the courses offered by the best ...,BOCCONI UNIVERSITY STRATEGY
3,"similarly, the fact that new generations are u...",BOCCONI UNIVERSITY STRATEGY
4,"as an example, today it is possible to produce...",BOCCONI UNIVERSITY STRATEGY
...,...,...
1665,this 200 billion yen will be used to support v...,University of TOKYO
1666,we know that achieving this goal will not be e...,University of TOKYO
1667,thank you for your continued support and dedic...,University of TOKYO
1668,the student supporters club scholarship contac...,University of TOKYO


Посмотрим распределение по университетам

In [70]:
df['university'].value_counts()

university
BOCCONI UNIVERSITY STRATEGY            216
University of TOKYO                    194
Saint Mary’s University                161
Florida International University       146
CAIP TOWN UNIVERSITY STRATEGY          131
STANFORD UNIVERSITY STRATEGY           119
university of manitoba                  90
Dawson College                          85
University of Georgia                   81
LONDON SCHOOL OF ECONOMICS STRATEGY     78
University of Maine at Augusta          55
COPENHAGEN UNIVERSITY STRATEGY          52
Texas Tech University                   52
University of Nebraska–Lincoln          48
University of Southern Mississippi      42
King’s University College               38
Montclair State University              33
University of Calgaryl                  25
Lakehead University                     19
University of Connecticut                5
Name: count, dtype: int64

# Анотация датасета

P.S. Я долго настриавал промт и валидировал руками качество на датасетах ``manual_test_0.csv`` и ``manual_test_1.csv`` в этой же папке

In [ ]:
client = OpenAI(api_key="") 

SYSTEM_PROMPT = """
Ты — эксперт в области институциональной социологии и дискурсивного анализа. Твоя задача — классифицировать отдельные предложения из стратегий университетов. Каждое предложение может быть помечено как:

- "1" — если оно содержит моральную легитимацию, то есть апеллирует к общественным ценностям, нормам, представлениям о должном, социальной миссии или этическим стандартам.
- "0" — если в предложении нет такого нормативного оправдания, а говорится о фактах, планах или выгодах без апелляции к моральным или общественным ценностям.

Моральная легитимация бывает четырёх типов (по Suchman, 1995):
1. Результативная — общественная польза, устойчивость, равенство.
2. Процедурная — справедливость, прозрачность, равные возможности.
3. Структурная — статус, миссия, долг.
4. Личностная — моральный авторитет лидеров.

Если есть хотя бы один из этих признаков — ставь "1", иначе "0". Отвечай только одной цифрой: 1 или 0.
""".strip()

def gpt_moral_classifier(text: str) -> int:
    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": text}
            ],
            max_tokens=1,
            temperature=0
        )
        answer = response.choices[0].message.content.strip()
        return int(answer) if answer in ("0", "1") else -1
    except Exception as e:
        print(f"Error on input: {text[:100]}... → {e}")
        return -1
 
def annotate_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    labels = []
    start_time = time.time()

    for sentence in tqdm(df['sentence'], desc="Annotating"):
        label = gpt_moral_classifier(sentence)
        labels.append(label)

    elapsed = time.time() - start_time
    print(f"\nTime per example: {elapsed / len(df):.2f} seconds")

    df = df.copy()
    df['moral_label'] = labels
    return df

df = annotate_dataframe(df)

Annotating:   0%|          | 0/1670 [00:00<?, ?it/s]


Time per example: 0.61 seconds


Посмотрим распредление классов

In [73]:
df['moral_label'].value_counts()

moral_label
0    942
1    728
Name: count, dtype: int64

Сохраним аннотированный датасет для дальнешей работы

In [74]:
df.to_csv('moral_legitimicy_annotated.csv')

Сохраним данные для мануального просмотра по классам

In [84]:
df[df['moral_label'] == 1][['sentence', 'moral_label']].sample(40).to_csv('manual_test_1.csv')

In [85]:
df[df['moral_label'] == 0][['sentence', 'moral_label']].sample(40).to_csv('manual_test_0.csv')